# Privacy and Security

## Overview

Modern machine learning systems are trained on data that is often sensitive: medical records, financial transactions, location traces, and private messages. A trained model can leak information about its training data through membership inference, model inversion, and memorization attacks. **Privacy-preserving machine learning** provides mathematical guarantees and engineering techniques that let us learn useful patterns from data while bounding what any individual record contributes to the released model.

This notebook covers the core building blocks: differential privacy and its mechanisms, private training with DP-SGD, federated learning, homomorphic encryption, and synthetic data generation.

---

## 1. Differential Privacy

Differential privacy (DP) is a formal definition of privacy. Two datasets $D$ and $D'$ are **neighboring** if they differ in exactly one record (one person added, removed, or changed). A randomized mechanism $M$ satisfies **$\epsilon$-differential privacy** if for all neighboring datasets $D, D'$ and all measurable output sets $S$:

$$\Pr[M(D) \in S] \le \exp(\epsilon) \, \Pr[M(D') \in S]$$

The parameter $\epsilon$ (the privacy budget) controls the strength of the guarantee: smaller $\epsilon$ means stronger privacy and more noise. The relaxed **$(\epsilon, \delta)$-differential privacy** allows a small failure probability $\delta$:

$$\Pr[M(D) \in S] \le \exp(\epsilon) \, \Pr[M(D') \in S] + \delta$$

Here $\delta$ should be cryptographically small, typically $\delta \ll 1/n$ where $n$ is the number of records.

---

## 2. Sensitivity

The amount of noise required depends on how much one record can change the query output. For a query $f$, the **$\ell_1$ sensitivity** is:

$$\Delta_1 f = \max_{D, D' \text{ neighbors}} \lVert f(D) - f(D') \rVert_1$$

and the **$\ell_2$ sensitivity** $\Delta_2 f$ uses the Euclidean norm. A counting query has $\Delta_1 f = 1$ (adding or removing one person changes the count by at most one).

---

## 3. The Laplace Mechanism

The **Laplace mechanism** achieves $\epsilon$-DP by adding noise drawn from a Laplace distribution with scale:

$$b = \frac{\Delta_1 f}{\epsilon}$$

The released output is $M(D) = f(D) + \mathrm{Lap}(b)$, where the Laplace density is $p(x) = \frac{1}{2b}\exp(-|x|/b)$. Larger sensitivity or smaller $\epsilon$ both increase the scale and therefore the noise.

---

## 4. The Gaussian Mechanism

The **Gaussian mechanism** adds zero-mean Gaussian noise calibrated to the $\ell_2$ sensitivity and satisfies $(\epsilon, \delta)$-DP when the standard deviation satisfies:

$$\sigma \ge \frac{\Delta_2 f \, \sqrt{2 \ln(1.25/\delta)}}{\epsilon}$$

The Gaussian mechanism composes more gracefully than Laplace and is the basis for DP-SGD.

---

## 5. Composition

If we run $k$ mechanisms, each $\epsilon_i$-DP, the **basic composition** theorem says the combined release is $\left(\sum_{i=1}^{k}\epsilon_i\right)$-DP. **Advanced composition** and the **moments accountant / Renyi DP** give much tighter bounds, roughly scaling like $\sqrt{k}$ rather than $k$, which is what makes thousands of private training steps feasible.

---

## 6. DP-SGD

**Differentially Private Stochastic Gradient Descent (DP-SGD)** makes training private with two changes per step:

1. **Per-example gradient clipping**: compute each example's gradient $g_i$ and rescale it so its $\ell_2$ norm is at most a clip bound $C$: $\bar{g}_i = g_i / \max(1, \lVert g_i \rVert_2 / C)$. This bounds the sensitivity of the gradient sum to $C$.
2. **Calibrated Gaussian noise**: add noise $\mathcal{N}(0, \sigma^2 C^2 I)$ to the summed clipped gradients, then average over the batch of size $B$:

$$\tilde{g} = \frac{1}{B}\left(\sum_{i} \bar{g}_i + \mathcal{N}(0, \sigma^2 C^2 I)\right)$$

The noise multiplier $\sigma$ together with the sampling rate and number of steps determines the total $(\epsilon, \delta)$ spent, tracked by a privacy accountant.

---

## 7. Federated Learning

In **federated learning** the raw data never leaves the client devices. Each client trains locally and sends only model updates to a server. **FedAvg** aggregates client weights as a weighted average:

$$w_{t+1} = \sum_{k=1}^{K} \frac{n_k}{n} \, w_t^{(k)}$$

where $n_k$ is client $k$'s sample count and $n = \sum_k n_k$. **Secure aggregation** uses cryptographic masking so the server learns only the sum of updates, never any individual client's update. DP noise can be added on top for end-to-end guarantees.

---

## 8. Homomorphic Encryption

**Homomorphic encryption (HE)** allows computation directly on ciphertexts. For a fully homomorphic scheme, $\mathrm{Dec}(\mathrm{Enc}(a) \oplus \mathrm{Enc}(b)) = a + b$ and similarly for multiplication, so a server can run inference on encrypted inputs and return an encrypted result without ever seeing the plaintext. HE is powerful but computationally expensive, so it is often combined with secure multiparty computation.

---

## 9. Synthetic Data Generation

**Synthetic data** replaces real records with artificial ones drawn from a model fit to the original distribution (GANs, VAEs, or statistical models). When the generator is trained with DP guarantees, the released synthetic dataset inherits those guarantees and can be shared freely. The challenge is preserving downstream utility and avoiding leakage of rare records.

---


In [1]:
import numpy as np

rng = np.random.default_rng(0)


def laplace_mechanism(true_value, sensitivity, epsilon, size=None):
    """Add Laplace noise with scale b = sensitivity / epsilon for epsilon-DP."""
    b = sensitivity / epsilon
    noise = rng.laplace(loc=0.0, scale=b, size=size)
    return true_value + noise


def gaussian_mechanism(true_value, sensitivity, epsilon, delta, size=None):
    """Add Gaussian noise calibrated for (epsilon, delta)-DP."""
    sigma = sensitivity * np.sqrt(2.0 * np.log(1.25 / delta)) / epsilon
    noise = rng.normal(loc=0.0, scale=sigma, size=size)
    return true_value + noise, sigma


# Private count query (sensitivity = 1 for add/remove one record).
data = rng.integers(0, 2, size=10000)  # binary attribute over 10000 people
true_count = data.sum()
print('True count:', true_count)

# Privacy-utility tradeoff: smaller epsilon -> more noise -> larger error.
print('\nPrivacy-utility tradeoff for a DP count query (sensitivity = 1):')
for eps in [0.1, 0.5, 1.0, 5.0]:
    trials = laplace_mechanism(true_count, sensitivity=1.0, epsilon=eps, size=2000)
    mae = np.mean(np.abs(trials - true_count))
    print(f'  epsilon = {eps:>4}:  scale b = {1.0/eps:6.2f}   mean abs error = {mae:7.2f}')

# Private mean query. For values bounded in [0, 1] over n records the mean
# has sensitivity 1/n. We use the Gaussian mechanism here.
n = len(data)
true_mean = data.mean()
priv_mean, sigma = gaussian_mechanism(true_mean, sensitivity=1.0 / n,
                                      epsilon=1.0, delta=1e-5)
print(f'\nGaussian DP mean (eps=1.0, delta=1e-5): true={true_mean:.4f} '
      f'private={priv_mean:.4f} sigma={sigma:.6f}')


True count: 5030

Privacy-utility tradeoff for a DP count query (sensitivity = 1):
  epsilon =  0.1:  scale b =  10.00   mean abs error =    9.98
  epsilon =  0.5:  scale b =   2.00   mean abs error =    2.00
  epsilon =  1.0:  scale b =   1.00   mean abs error =    0.98
  epsilon =  5.0:  scale b =   0.20   mean abs error =    0.20

Gaussian DP mean (eps=1.0, delta=1e-5): true=0.5030 private=0.5031 sigma=0.000484


In [2]:
import numpy as np


def dp_sgd_step(per_example_grads, clip_bound, noise_multiplier, lr,
                weights, rng):
    """One DP-SGD update.

    per_example_grads : array of shape (batch, dim), one gradient per example.
    clip_bound (C)    : maximum allowed L2 norm of each per-example gradient.
    noise_multiplier  : Gaussian noise std as a multiple of C.
    Returns updated weights and the noisy averaged gradient.
    """
    g = np.asarray(per_example_grads, dtype=float)
    batch_size, dim = g.shape

    # 1. Per-example gradient clipping: rescale so each norm <= clip_bound.
    norms = np.linalg.norm(g, axis=1, keepdims=True)
    scale = np.maximum(1.0, norms / clip_bound)
    clipped = g / scale

    # 2. Sum clipped gradients and add calibrated Gaussian noise.
    summed = clipped.sum(axis=0)
    noise = rng.normal(0.0, noise_multiplier * clip_bound, size=dim)
    noisy_avg_grad = (summed + noise) / batch_size

    # 3. Gradient descent update.
    new_weights = weights - lr * noisy_avg_grad
    return new_weights, noisy_avg_grad


rng = np.random.default_rng(1)
dim = 5
weights = np.zeros(dim)
batch_grads = rng.normal(0.0, 2.0, size=(32, dim))  # raw per-example grads

new_weights, noisy_grad = dp_sgd_step(batch_grads, clip_bound=1.0,
                                      noise_multiplier=1.1, lr=0.1,
                                      weights=weights, rng=rng)
print('Noisy averaged gradient:', np.round(noisy_grad, 4))
print('Updated weights:        ', np.round(new_weights, 4))

# Show clipping in action.
raw_norms = np.linalg.norm(batch_grads, axis=1)
clipped_norms = np.minimum(raw_norms, 1.0)
print(f'\nRaw grad norm range:     [{raw_norms.min():.2f}, {raw_norms.max():.2f}]')
print(f'Clipped grad norm range: [{clipped_norms.min():.2f}, {clipped_norms.max():.2f}]')


Noisy averaged gradient: [ 0.0386 -0.0719 -0.192  -0.0865  0.0965]
Updated weights:         [-0.0039  0.0072  0.0192  0.0086 -0.0096]

Raw grad norm range:     [1.65, 8.39]
Clipped grad norm range: [1.00, 1.00]


In [3]:
import numpy as np


def fed_avg(client_weights, client_counts=None):
    """FedAvg aggregation of client weight arrays.

    client_weights : list of equally shaped numpy arrays (one per client).
    client_counts  : optional list of sample counts n_k for weighting.
                     If None, a plain (uniform) average is returned.
    """
    stacked = np.stack([np.asarray(w, dtype=float) for w in client_weights])
    if client_counts is None:
        return stacked.mean(axis=0)
    counts = np.asarray(client_counts, dtype=float)
    fractions = counts / counts.sum()
    # Weighted sum: w = sum_k (n_k / n) * w_k
    shape = (-1,) + (1,) * (stacked.ndim - 1)
    return (stacked * fractions.reshape(shape)).sum(axis=0)


# Three clients each holding a (3,) weight vector.
client_weights = [
    np.array([1.0, 2.0, 3.0]),
    np.array([2.0, 0.0, 4.0]),
    np.array([3.0, 4.0, 5.0]),
]
client_counts = [100, 50, 850]

print('Uniform FedAvg: ', fed_avg(client_weights))
print('Weighted FedAvg:', fed_avg(client_weights, client_counts))


Uniform FedAvg:  [2. 2. 4.]
Weighted FedAvg: [2.75 3.6  4.75]


In [4]:
# Opacus PrivacyEngine usage (reference only, requires PyTorch).
# pip install opacus
#
# import torch
# from torch import nn
# from opacus import PrivacyEngine
#
# model = nn.Linear(20, 2)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
# data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256)
#
# privacy_engine = PrivacyEngine()
# model, optimizer, data_loader = privacy_engine.make_private(
#     module=model,
#     optimizer=optimizer,
#     data_loader=data_loader,
#     noise_multiplier=1.1,   # sigma: Gaussian noise std as a multiple of C
#     max_grad_norm=1.0,      # C: per-example gradient clipping bound
# )
#
# # Train as usual; Opacus handles per-example clipping and noise internally.
# for x, y in data_loader:
#     optimizer.zero_grad()
#     loss = nn.functional.cross_entropy(model(x), y)
#     loss.backward()
#     optimizer.step()
#
# # Query the spent privacy budget after each epoch.
# epsilon = privacy_engine.get_epsilon(delta=1e-5)
# print(f'(epsilon = {epsilon:.2f}, delta = 1e-5)-DP so far')

print('Opacus block is reference only; install opacus and PyTorch to run it.')


Opacus block is reference only; install opacus and PyTorch to run it.


---

## Additional Learning Resources

### Papers

- The Algorithmic Foundations of Differential Privacy (Dwork & Roth): https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf
- How to DP-fy ML: A Practical Guide to Machine Learning with Differential Privacy: https://arxiv.org/abs/2108.07345
- Deep Learning with Differential Privacy (DP-SGD, Abadi et al.): https://arxiv.org/abs/1607.00133

### Books & Courses

- Programming Differential Privacy (open textbook): https://programming-dp.com/

### Tools & Libraries

- Opacus (DP-SGD for PyTorch): https://github.com/pytorch/opacus
- TensorFlow Privacy: https://github.com/tensorflow/privacy
- Diffprivlib (IBM Differential Privacy Library): https://github.com/IBM/differential-privacy-library
- PySyft (federated learning and privacy): https://github.com/OpenMined/PySyft
